# Transformer：从注意力到最小可训练 Decoder

本 notebook 用 PyTorch 从张量形状出发搭建一个可训练的 decoder-only Transformer。代码刻意保持小规模，便于在 CPU 上逐格运行和断点观察。

## 学习目标

1. 能写出 scaled dot-product attention，并解释 Q、K、V、缩放与 causal mask。
2. 能沿着 `[batch, sequence, model_dim]` 追踪拆头、注意力和合头形状。
3. 能区分 attention 的 token 混合与 FFN 的通道变换，理解 Pre-Norm、残差、RMSNorm 与 SwiGLU。
4. 能完成标签右移、交叉熵、反向传播和最小贪心推理。
5. 能说清训练、prefill、decode 的计算差异和常见工程错误。


In [ ]:
import math  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

torch.manual_seed(7)  # 执行当前语句以推进本节示例。
torch.set_printoptions(precision=4, sci_mode=False)  # 计算并保存当前步骤的中间状态。
device = torch.device("cpu")  # 计算并保存当前步骤的中间状态。
print("PyTorch:", torch.__version__, "device:", device)  # 执行当前语句以推进本节示例。


## 1. 架构与核心公式

现代 decoder-only block 常采用 Pre-Norm：

\[H=X+\operatorname{MHA}(\operatorname{Norm}(X)),\qquad Y=H+\operatorname{FFN}(\operatorname{Norm}(H)).\]

单头 scaled dot-product attention 为：

\[Q=XW_Q,\ K=XW_K,\ V=XW_V,\qquad A=\operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_h}}+M\right),\quad O=AV.\]

`M` 在未来位置填负无穷。除以 `sqrt(d_h)` 是为了让点积方差不随 head dimension 线性增大，避免 softmax 过早饱和。训练时虽然使用 causal mask，所有 query 位置仍可并行计算。


In [ ]:
def scaled_dot_product_attention(q, k, v, causal=False, padding_mask=None):  # 定义本节可复用的核心函数。
    # q: [B, Hq, Tq, Dh], k/v: [B, Hkv, Tk, Dh] 中文说明：该注释解释本行约束。
    scores = q @ k.transpose(-2, -1) / math.sqrt(q.size(-1))  # 计算并保存当前步骤的中间状态。
    if causal:  # 按当前条件选择后续控制路径。
        tq, tk = q.size(-2), k.size(-2)  # 计算并保存当前步骤的中间状态。
        # full self-attention 演示：query i 只能看到 key <= i
        mask = torch.ones(tq, tk, dtype=torch.bool, device=q.device).triu(1)  # 计算并保存当前步骤的中间状态。
        scores = scores.masked_fill(mask, float("-inf"))  # 计算并保存当前步骤的中间状态。
    if padding_mask is not None:  # 按当前条件选择后续控制路径。
        # padding_mask: [B, Tk]，True 表示有效 key
        scores = scores.masked_fill(~padding_mask[:, None, None, :], float("-inf"))  # 计算并保存当前步骤的中间状态。
    probs = torch.softmax(scores.float(), dim=-1).to(q.dtype)  # 计算并保存当前步骤的中间状态。
    return probs @ v, probs  # 返回当前分支计算出的结果。

B, H, S, Dh = 2, 3, 4, 5  # 计算并保存当前步骤的中间状态。
q = torch.randn(B, H, S, Dh)  # 计算并保存当前步骤的中间状态。
k = torch.randn(B, H, S, Dh)  # 计算并保存当前步骤的中间状态。
v = torch.randn(B, H, S, Dh)  # 计算并保存当前步骤的中间状态。
out, weights = scaled_dot_product_attention(q, k, v, causal=True)  # 计算并保存当前步骤的中间状态。
print("q/k/v:", q.shape, k.shape, v.shape)  # 执行当前语句以推进本节示例。
print("attention weights:", weights.shape, "output:", out.shape)  # 执行当前语句以推进本节示例。
print("第 0 个 query 的未来权重：", weights[0, 0, 0, 1:])  # 执行当前语句以推进本节示例。
assert torch.equal(weights[0, 0, 0, 1:], torch.zeros(S - 1))  # 用受控断言验证关键不变量。


### 一个可手算的注意力例子

下面让一个 query 与两个 keys 的缩放前点积分别为 1 和 0。softmax 决定从两个 values 取多少，而不是把 key 本身相加。这个例子也说明 Q/K 负责匹配，V 负责被汇聚的内容。


In [ ]:
q_small = torch.tensor([[[[1.0, 0.0]]]])          # [1,1,1,2]；中文说明：该行遵循既定约束。
k_small = torch.tensor([[[[1.0, 0.0], [0.0, 1.0]]]])  # 计算并保存当前步骤的中间状态。
v_small = torch.tensor([[[[1.0, 0.0], [0.0, 2.0]]]])  # 计算并保存当前步骤的中间状态。
o_small, p_small = scaled_dot_product_attention(q_small, k_small, v_small)  # 计算并保存当前步骤的中间状态。
print("softmax 权重:", p_small.flatten())  # 执行当前语句以推进本节示例。
print("加权后的 value:", o_small.flatten())  # 执行当前语句以推进本节示例。
print("每行概率和:", p_small.sum(dim=-1))  # 计算并保存当前步骤的中间状态。


## 2. 多头注意力与形状追踪

输入 `X: [B,S,D]` 经一次融合投影得到 QKV，再 reshape 为 `[B,H,S,Dh]`，其中 `D=H*Dh`。每个 head 有独立的 softmax 路由图；各头输出合并回 `[B,S,D]` 后才能与 residual 相加。多头不是把计算机械乘 H 倍：总宽度 D 固定时，投影参数与主导 FLOPs 仍同阶。


In [ ]:
class MultiHeadSelfAttention(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, n_heads):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        assert d_model % n_heads == 0  # 用受控断言验证关键不变量。
        self.n_heads = n_heads  # 计算并保存当前步骤的中间状态。
        self.head_dim = d_model // n_heads  # 计算并保存当前步骤的中间状态。
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)  # 计算并保存当前步骤的中间状态。
        self.out_proj = nn.Linear(d_model, d_model, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x, return_shapes=False):  # 定义本节可复用的核心函数。
        b, s, d = x.shape  # 计算并保存当前步骤的中间状态。
        qkv = self.qkv(x).view(b, s, 3, self.n_heads, self.head_dim)  # 计算并保存当前步骤的中间状态。
        q, k, v = qkv.unbind(dim=2)  # 计算并保存当前步骤的中间状态。
        q, k, v = (t.transpose(1, 2) for t in (q, k, v))  # 计算并保存当前步骤的中间状态。
        head_out, probs = scaled_dot_product_attention(q, k, v, causal=True)  # 计算并保存当前步骤的中间状态。
        merged = head_out.transpose(1, 2).contiguous().view(b, s, d)  # 计算并保存当前步骤的中间状态。
        y = self.out_proj(merged)  # 计算并保存当前步骤的中间状态。
        if return_shapes:  # 按当前条件选择后续控制路径。
            return y, {"input": x.shape, "q": q.shape, "scores": probs.shape,  # 返回当前分支计算出的结果。
                       "heads_out": head_out.shape, "merged": merged.shape}  # 执行当前语句以推进本节示例。
        return y  # 返回当前分支计算出的结果。

mha = MultiHeadSelfAttention(d_model=16, n_heads=4)  # 计算并保存当前步骤的中间状态。
x_demo = torch.randn(2, 6, 16)  # 计算并保存当前步骤的中间状态。
y_demo, shape_trace = mha(x_demo, return_shapes=True)  # 计算并保存当前步骤的中间状态。
for name, shape in shape_trace.items():  # 遍历输入元素以累积或检查结果。
    print(f"{name:10s} -> {tuple(shape)}")  # 执行当前语句以推进本节示例。
assert y_demo.shape == x_demo.shape  # 用受控断言验证关键不变量。


## 3. RMSNorm、SwiGLU、残差与 Pre-Norm block

RMSNorm 对每个 token 的 hidden 维按均方根缩放：

\[\operatorname{RMSNorm}(x)=\gamma\odot x / \sqrt{\frac{1}{d}\sum_i x_i^2+\epsilon}.\]

SwiGLU 使用一条 gate 分支调制 up 分支：

\[\operatorname{SwiGLU}(x)=W_o(\operatorname{SiLU}(W_gx)\odot W_ux).\]

残差提供恒等梯度路径；Norm 控制输入尺度；两者职责不同。下面的 block 保证两个子层输出都回到 `d_model`。


In [ ]:
class RMSNorm(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, dim, eps=1e-6):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.weight = nn.Parameter(torch.ones(dim))  # 计算并保存当前步骤的中间状态。
        self.eps = eps  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        scale = torch.rsqrt(x.float().pow(2).mean(dim=-1, keepdim=True) + self.eps)  # 计算并保存当前步骤的中间状态。
        return (x * scale.to(x.dtype)) * self.weight  # 返回当前分支计算出的结果。

class SwiGLU(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, d_ff):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.gate = nn.Linear(d_model, d_ff, bias=False)  # 计算并保存当前步骤的中间状态。
        self.up = nn.Linear(d_model, d_ff, bias=False)  # 计算并保存当前步骤的中间状态。
        self.down = nn.Linear(d_ff, d_model, bias=False)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        return self.down(F.silu(self.gate(x)) * self.up(x))  # 返回当前分支计算出的结果。

class TransformerBlock(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, d_model, n_heads, d_ff):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.norm1 = RMSNorm(d_model)  # 计算并保存当前步骤的中间状态。
        self.attn = MultiHeadSelfAttention(d_model, n_heads)  # 计算并保存当前步骤的中间状态。
        self.norm2 = RMSNorm(d_model)  # 计算并保存当前步骤的中间状态。
        self.ffn = SwiGLU(d_model, d_ff)  # 计算并保存当前步骤的中间状态。

    def forward(self, x):  # 定义本节可复用的核心函数。
        x = x + self.attn(self.norm1(x))  # 计算并保存当前步骤的中间状态。
        return x + self.ffn(self.norm2(x))  # 返回当前分支计算出的结果。

block = TransformerBlock(16, 4, 32)  # 计算并保存当前步骤的中间状态。
block_out = block(x_demo)  # 计算并保存当前步骤的中间状态。
print("block input/output:", x_demo.shape, block_out.shape)  # 执行当前语句以推进本节示例。
print("RMS after norm:", RMSNorm(16)(x_demo).float().pow(2).mean(-1).sqrt().mean().item())  # 执行当前语句以推进本节示例。


## 4. 最小可训练 decoder 与标签右移

因果语言模型最大化 `p(x_t | x_<t)`。若输入序列为 `<bos> 我 爱 NLP <eos>`，位置 0 的 logits 预测“我”，位置 1 预测“爱”。代码里使用 `logits[:, :-1]` 对齐 `tokens[:, 1:]`。标签右移不能替代 causal mask：若 mask 错了，模型仍可偷看答案。


In [ ]:
class TinyDecoder(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self, vocab_size=32, d_model=32, n_heads=4, d_ff=64, n_layers=2):  # 定义本节可复用的核心函数。
        super().__init__()  # 执行当前语句以推进本节示例。
        self.embed = nn.Embedding(vocab_size, d_model)  # 计算并保存当前步骤的中间状态。
        self.blocks = nn.ModuleList([  # 计算并保存当前步骤的中间状态。
            TransformerBlock(d_model, n_heads, d_ff) for _ in range(n_layers)  # 执行当前语句以推进本节示例。
        ])  # 执行当前语句以推进本节示例。
        self.final_norm = RMSNorm(d_model)  # 计算并保存当前步骤的中间状态。
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)  # 计算并保存当前步骤的中间状态。
        self.lm_head.weight = self.embed.weight  # weight tying；中文说明：该行遵循既定约束。

    def forward(self, tokens):  # 定义本节可复用的核心函数。
        x = self.embed(tokens)  # 计算并保存当前步骤的中间状态。
        for layer in self.blocks:  # 遍历输入元素以累积或检查结果。
            x = layer(x)  # 计算并保存当前步骤的中间状态。
        return self.lm_head(self.final_norm(x))  # 返回当前分支计算出的结果。

model = TinyDecoder().to(device)  # 计算并保存当前步骤的中间状态。
tokens = torch.tensor([[1, 5, 9, 5, 2], [1, 4, 7, 8, 2]], device=device)  # 计算并保存当前步骤的中间状态。
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3)  # 计算并保存当前步骤的中间状态。
for step in range(6):  # 遍历输入元素以累积或检查结果。
    logits = model(tokens)  # 计算并保存当前步骤的中间状态。
    loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)), tokens[:, 1:].reshape(-1))  # 计算并保存当前步骤的中间状态。
    optimizer.zero_grad()  # 执行当前语句以推进本节示例。
    loss.backward()  # 执行当前语句以推进本节示例。
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # 执行当前语句以推进本节示例。
    optimizer.step()  # 执行当前语句以推进本节示例。
    print(f"step={step} loss={loss.item():.4f}")  # 计算并保存当前步骤的中间状态。
assert torch.isfinite(loss)  # 用受控断言验证关键不变量。


## 5. 最小贪心推理与三种计算阶段

下面为了保持代码短小，每步重新跑完整前缀；生产推理会缓存每层历史 K/V。三种阶段要分开分析：

| 阶段 | 输入形态 | 主要特征 | 常用指标 |
|---|---|---|---|
| 训练 | 完整序列 + 反向 | 大 GEMM、保存/重算激活 | tokens/s、MFU |
| prefill | 完整 prompt 前向 | 长序列 attention、建立 KV | TTFT、prompt tokens/s |
| decode | 每请求一个新 token | 逐步串行、权重/KV 带宽 | TPOT、ITL、output tokens/s |

causal mask 不会阻止训练并行；它只限制可见性。decode 串行是因为下一个 token 输入依赖当前采样结果。


In [ ]:
@torch.no_grad()  # 为下方定义附加声明式配置。
def greedy_generate(model, prompt, max_new_tokens=4):  # 定义本节可复用的核心函数。
    model.eval()  # 执行当前语句以推进本节示例。
    generated = prompt.clone()  # 计算并保存当前步骤的中间状态。
    for _ in range(max_new_tokens):  # 遍历输入元素以累积或检查结果。
        next_token = model(generated)[:, -1].argmax(dim=-1, keepdim=True)  # 计算并保存当前步骤的中间状态。
        generated = torch.cat([generated, next_token], dim=1)  # 计算并保存当前步骤的中间状态。
    return generated  # 返回当前分支计算出的结果。

prompt = torch.tensor([[1, 5]], device=device)  # 计算并保存当前步骤的中间状态。
print("prompt   :", prompt.tolist())  # 执行当前语句以推进本节示例。
print("generated:", greedy_generate(model, prompt).tolist())  # 执行当前语句以推进本节示例。


## 6. 变体对比与工程坑

| 主题 | 常见选择 | 关键取舍 |
|---|---|---|
| 架构 | encoder / decoder / encoder-decoder | 双向表征、因果生成、cross-attention |
| Norm | LayerNorm / RMSNorm | 去均值能力 vs 更简单归约 |
| Norm 位置 | Post-Norm / Pre-Norm | 原始结构 vs 深层训练稳定性 |
| FFN | GELU / SwiGLU / MoE | 参数与门控能力、条件计算与通信 |
| Attention | MHA / GQA / MQA | KV 多样性 vs cache/带宽 |
| 长序列 | 标准 / Flash / sparse | 精确 IO 优化 vs 改变连接模式 |

高频坑：QKV reshape 后 transpose 轴错；忘除 `sqrt(dh)`；mask True/False 语义反；一行全被 mask 导致 NaN；把 padding mask 当 loss mask；标签重复 shift；低精度直接做 norm/softmax 归约；训练时误以为 causal attention 必须逐 token 循环。调试优先做单 batch 过拟合，并验证“修改未来 token 不影响过去 logits”。


## 7. 练习与面试总结

### 动手练习

1. 给 `scaled_dot_product_attention` 增加左 padding 样本，验证有效位置与单独运行一致。
2. 将 Pre-Norm 改成 Post-Norm，比较 8 层小模型的梯度范数。
3. 实现 GQA：令 `Hq=4,Hkv=2`，避免物理复制 K/V。
4. 给 `TinyDecoder` 增加 dropout 与 padding loss mask，并做一个 batch 过拟合测试。
5. 用框架 fused SDPA 替换手写 attention，对比输出误差和速度。

### 面试时的 60 秒主线

Transformer block = attention 跨 token 路由 + FFN 逐 token 变换 + residual/norm 稳定深层训练。QK 点积经 `sqrt(dh)` 缩放和 mask 后 softmax，再对 V 加权；多头提供多张独立路由图。训练用 teacher forcing 并行计算、标签右移求交叉熵；prefill 批量建状态，decode 逐 token 且常受带宽限制。最后主动指出标准 attention 的二次长序列成本、混合精度和 mask 正确性。
